#**LAB 8 (21 Mar- 22 Mar 2025)**
#TOPICS COVERED : Recurrent Neural Networks


In [70]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np


[https://karpathy.github.io/2015/05/21/rnn-effectiveness/](https://karpathy.github.io/2015/05/21/rnn-effectiveness/)

The notebook provides a comprehensive overview of building and training a basic RNN for a character-level language modeling task. It covers data preprocessing, model definition, training, and using the model for predictions.

In [71]:
text = ['hey how are you', 'good i am fine', 'have a nice day']
text1=['i like cake','she likes Pizza','she loves flower']

In [74]:
chars = set(''.join(text))
chars

{' ',
 'a',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'm',
 'n',
 'o',
 'r',
 'u',
 'v',
 'w',
 'y'}

In [76]:
chars1 = set(''.join(text1))
chars1

{' ',
 'P',
 'a',
 'c',
 'e',
 'f',
 'h',
 'i',
 'k',
 'l',
 'o',
 'r',
 's',
 'v',
 'w',
 'z'}

In [77]:
int_map = dict(enumerate(chars))
int_map

{0: 'v',
 1: 'c',
 2: 'o',
 3: 'y',
 4: 'a',
 5: 'r',
 6: 'n',
 7: 'e',
 8: 'h',
 9: 'w',
 10: ' ',
 11: 'g',
 12: 'd',
 13: 'm',
 14: 'u',
 15: 'i',
 16: 'f'}

In [78]:
int_map1 = dict(enumerate(chars1))
int_map1

{0: 'c',
 1: 'v',
 2: 'a',
 3: 'o',
 4: 'r',
 5: 's',
 6: 'i',
 7: 'P',
 8: 'e',
 9: 'h',
 10: 'w',
 11: ' ',
 12: 'z',
 13: 'k',
 14: 'l',
 15: 'f'}

In [79]:
char_map = {char:ind for ind, char in int_map.items()}
char_map

{'v': 0,
 'c': 1,
 'o': 2,
 'y': 3,
 'a': 4,
 'r': 5,
 'n': 6,
 'e': 7,
 'h': 8,
 'w': 9,
 ' ': 10,
 'g': 11,
 'd': 12,
 'm': 13,
 'u': 14,
 'i': 15,
 'f': 16}

In [80]:
char_map1 = {char1:ind for ind, char1 in int_map1.items()}
char_map1

{'c': 0,
 'v': 1,
 'a': 2,
 'o': 3,
 'r': 4,
 's': 5,
 'i': 6,
 'P': 7,
 'e': 8,
 'h': 9,
 'w': 10,
 ' ': 11,
 'z': 12,
 'k': 13,
 'l': 14,
 'f': 15}

In [9]:
num_unique_chars = len(char_map)
num_unique_chars

In [10]:
num_unique_chars

17

In [11]:
maxlen = len(max(text, key=len))

In [12]:
maxlen

15

# the items must be of the same dims if we want to stack them into a batch

for example, say we have a dataset of images

and say our batch size is 30

256x256

512x512

(30, 3, 256, 256)

In [13]:
# iterating over my sentences in the dataset
for i in range(len(text)):
  while(len(text[i]))<maxlen:
    text[i] += ' '

In [14]:
text

['hey how are you', 'good i am fine ', 'have a nice day']

In [15]:
input_seq = list()
target_seq = list()


for i in range(len(text)):
  input_seq.append(text[i][:-1])##REMOVE LAST CHARACTER
  target_seq.append(text[i][1:])##REMOVE FIRST CHARACTER


In [16]:
input_seq

['hey how are yo', 'good i am fine', 'have a nice da']

In [17]:
target_seq

['ey how are you', 'ood i am fine ', 'ave a nice day']

In [18]:
for i in range(len(text)):
  input_seq[i] = [char_map[character] for character in input_seq[i]]
  target_seq[i] = [char_map[character] for character in target_seq[i]]


In [19]:
input_seq


[[8, 7, 3, 10, 8, 2, 9, 10, 4, 5, 7, 10, 3, 2],
 [11, 2, 2, 12, 10, 15, 10, 4, 13, 10, 16, 15, 6, 7],
 [8, 4, 0, 7, 10, 4, 10, 6, 15, 1, 7, 10, 12, 4]]

In [20]:
target_seq


[[7, 3, 10, 8, 2, 9, 10, 4, 5, 7, 10, 3, 2, 14],
 [2, 2, 12, 10, 15, 10, 4, 13, 10, 16, 15, 6, 7, 10],
 [4, 0, 7, 10, 4, 10, 6, 15, 1, 7, 10, 12, 4, 3]]

In [21]:
# you want to get the one-hot embedding/vector corresponding to 4

[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [22]:
import numpy as np

import torch
from torch import nn


num_of_sentences x num_characters_in_sentence x length_of_one_hot_vector

In [23]:
def one_hot_encode(sequence, num_unique_chars, seq_len, batch_size):

  features = np.zeros((batch_size, seq_len, num_unique_chars), dtype=np.float32)
  # for each sentence
  for i in range(batch_size):
    # for each character in a sentence
    for u in range(seq_len):
      features[i, u, sequence[i][u]] = 1

  return features


In [24]:
batch_size = len(text)
seq_len = maxlen - 1

In [25]:
input_seq = one_hot_encode(input_seq, num_unique_chars, seq_len, batch_size)

In [26]:
type(input_seq)

numpy.ndarray

In [27]:
input_seq = torch.from_numpy(input_seq)
target_seq = torch.Tensor(target_seq)

In [28]:
device = torch.device('cuda')

# Making the model

In [29]:
class Model(nn.Module):
  def __init__(self, input_size, output_size, hidden_dim, n_layers):
    super().__init__()
    self.hidden_dim = hidden_dim
    self.n_layers = n_layers

    self.rnn = nn.RNN(input_size, hidden_dim, n_layers, batch_first = True)
    self.fc = nn.Linear(hidden_dim, output_size)

  def init_hidden(self, batch_size):
    hidden = torch.zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
    return hidden

  def forward(self, x):
    batch_size = x.shape[0]
    hidden = self.init_hidden(batch_size)

    out, hidden  = self.rnn(x, hidden)

    out = self.fc(out)

    return out, hidden


In [30]:
model = Model(input_size = num_unique_chars, output_size = num_unique_chars, hidden_dim = 12, n_layers = 1)

In [31]:
model = model.to(device)

In [32]:
n_epochs = 100
lr = 0.01

In [33]:
loss = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [34]:
input_seq = input_seq.to(device)

# Training the model

In [35]:
for epoch in range(1, n_epochs + 1):
  optimizer.zero_grad()

  output, hidden = model(input_seq)

  output = output.to(device)
  target_seq = target_seq.to(device)

  epoch_loss = loss(output.view(-1, output.shape[-1]), target_seq.view(-1).long())

  epoch_loss.backward()
  optimizer.step()

  if epoch % 10 == 0:
    print("Epoch: {}/{}............".format(epoch, n_epochs), end = ' ')
    print("Loss: {:.4f}".format(epoch_loss.item()))

Epoch: 10/100............ Loss: 2.4017
Epoch: 20/100............ Loss: 2.1173
Epoch: 30/100............ Loss: 1.7412
Epoch: 40/100............ Loss: 1.3421
Epoch: 50/100............ Loss: 0.9906
Epoch: 60/100............ Loss: 0.7092
Epoch: 70/100............ Loss: 0.4974
Epoch: 80/100............ Loss: 0.3508
Epoch: 90/100............ Loss: 0.2506
Epoch: 100/100............ Loss: 0.1862


In [36]:
output, hidden = model(input_seq)

In [37]:
output.shape

torch.Size([3, 14, 17])

In [38]:
target_seq.shape

torch.Size([3, 14])

In [39]:
target_seq.view(-1)

tensor([ 7.,  3., 10.,  8.,  2.,  9., 10.,  4.,  5.,  7., 10.,  3.,  2., 14.,
         2.,  2., 12., 10., 15., 10.,  4., 13., 10., 16., 15.,  6.,  7., 10.,
         4.,  0.,  7., 10.,  4., 10.,  6., 15.,  1.,  7., 10., 12.,  4.,  3.],
       device='cuda:0')

# Get predictions from our trained model

In [40]:
# characters = ['h', 'e', 'y']
def predict(model, characters):
  characters = np.array([[char_map[c] for c in characters]])
  characters = one_hot_encode(characters, num_unique_chars, characters.shape[1], 1)
  characters = torch.from_numpy(characters)
  characters = characters.to(device)

  model.eval()

  out, hidden = model(characters)

  prob = nn.functional.softmax(torch.squeeze(out, dim=0)[-1], dim=0)

  char_ind = torch.argmax(prob, dim=0)

  return int_map[char_ind.item()], hidden

In [41]:
def sample(model, out_len, start):

  model.eval()

  start = start.lower()

  chars = [ch for ch in start]

  size = out_len - len(chars)

  for _ in range(size):
    char, h = predict(model, chars)
    chars.append(char)

  return ''.join(chars)

In [42]:
sample(model, 15, 'have')

'have a nice day'

In [43]:
# [0.01, 0.02, 0.3, 0.04, .....]
# you apply softmax to this -> you get a list of probabilites
# you use torch.argmax to get the index corresponding to the max value
# say our max value -> 3
# we use the int_map to convert 3 to 'i'


Your task is to modify the existiong character level RNN model to create a word-level language model. Instead of predicting the next character you model should predict next word in a sequence of words.

#STEPS:
1. Data Preprocessing: Use larger text dataset(use any text corpus available in the lab). Tokenize the text data into words instead of chars. Create mapping of each unique word to an integer(word to index) and the reverse mapping(index to word.
2.Model Modification: Adjust the input and output dimensions of your RNN model to accommodate the size of the word vocabulary /9no of unique words0. Consider experimenting with the size of the hidden layer or adding more layers to the RNN.
3. Training
4.Evaluation and Generation

In [44]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [45]:
# Sample text data
text = ['hey how are you', 'good i am fine', 'have a nice day']

In [46]:
# Tokenize at word level
words = set(" ".join(text).split())
word2idx = {word: idx for idx, word in enumerate(words)}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(word2idx)

In [47]:
# Convert sentences to sequences
sequences = [[word2idx[word] for word in sentence.split()] for sentence in text]

In [48]:
# Create input-output pairs
inputs = []
targets = []
for seq in sequences:
    for i in range(len(seq) - 1):
        inputs.append(seq[:i+1])
        targets.append(seq[i+1])

In [49]:
# Padding sequences to have the same length
max_len = max(len(seq) for seq in inputs)
inputs = [seq + [0] * (max_len - len(seq)) for seq in inputs]

tensor_inputs = torch.tensor(inputs, dtype=torch.long)
tensor_targets = torch.tensor(targets, dtype=torch.long)


In [50]:
# Dataset class
class WordDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

In [51]:
# DataLoader
dataset = WordDataset(tensor_inputs, tensor_targets)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [52]:
# Define Word-Level RNN model
class WordLevelRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(WordLevelRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        out = self.fc(hidden.squeeze(0))
        return out

In [53]:
# Model, Loss, Optimizer
embedding_dim = 10
hidden_dim = 20
model = WordLevelRNN(vocab_size, embedding_dim, hidden_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [54]:
# Training loop
epochs = 100
for epoch in range(epochs):
    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

Epoch 0, Loss: 2.778557300567627
Epoch 10, Loss: 0.649502694606781
Epoch 20, Loss: 0.08154980838298798
Epoch 30, Loss: 0.03159413859248161
Epoch 40, Loss: 0.01946081779897213
Epoch 50, Loss: 1.0049036741256714
Epoch 60, Loss: 0.7656551599502563
Epoch 70, Loss: 0.004929176066070795
Epoch 80, Loss: 0.006029156036674976
Epoch 90, Loss: 0.005832438822835684


In [55]:

# Prediction function
def predict_next_word(model, sentence, word2idx, idx2word):
    words = sentence.split()
    seq = [word2idx[word] for word in words if word in word2idx]
    seq = torch.tensor([seq + [0] * (max_len - len(seq))], dtype=torch.long)
    output = model(seq)
    predicted_idx = torch.argmax(output, dim=1).item()
    return idx2word[predicted_idx]

In [56]:
# Example usage
print(predict_next_word(model, "hey how", word2idx, idx2word))

are


In [57]:
# Example usage
print(predict_next_word(model, "hey how are", word2idx, idx2word))

are


In [58]:
# Example usage
print(predict_next_word(model, "hey how are you ", word2idx, idx2word))

day


In [59]:
# Example usage
print(predict_next_word(model, "hey how are you  are", word2idx, idx2word))

am


Train the model with the learning rate 0.001, Epoch = 500, optimizer = SGD. Add one more RNN layer.

In [ ]:
# prompt: Train the model with the learning rate 0.001, Epoch = 500, optimizer = SGD. Add one more RNN layer. on a new data text1=['i like cake','she likes Pizza','she loves flower']

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torch import nn

# ... (Your existing code for data preprocessing and model definition) ...

# Define Word-Level RNN model with two RNN layers
class WordLevelRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(WordLevelRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn1 = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.rnn2 = nn.RNN(hidden_dim, hidden_dim, batch_first=True)  # Add another RNN layer
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn1(embedded)
        output, hidden = self.rnn2(output, hidden)  # Pass output and hidden state from rnn1 to rnn2
        out = self.fc(hidden.squeeze(0))
        return out


# Model, Loss, Optimizer
embedding_dim = 10
hidden_dim = 20
model = WordLevelRNN(vocab_size, embedding_dim, hidden_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)  # Use SGD optimizer with learning rate 0.001

# Training loop
epochs = 500  # Set epochs to 500
for epoch in range(epochs):
    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    if epoch % 50 == 0:  # Print loss every 50 epochs
        print(f'Epoch {epoch}, Loss: {loss.item()}')

# ... (Your existing code for prediction function) ...


In [65]:
embedding_dim = 10
hidden_dim = 20
n_layers = 2  # Add one more RNN layer
model = WordLevelRNNN(vocab_size, embedding_dim, hidden_dim, n_layers)  # Update model initialization
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)  # Use SGD with learning rate 0.001


TypeError: super(type, obj): obj must be an instance or subtype of type

In [66]:
# Updated WordLevelRNN class to include multiple RNN layers
class WordLevelRNNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers=1):
        super(WordLevelRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        out = self.fc(hidden[-1])  # Take the output from the last layer
        return out


In [67]:
# Training loop
epochs = 500  # Set epochs to 500
for epoch in range(epochs):
    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    if epoch % 50 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')


Epoch 0, Loss: 0.005386007949709892
Epoch 50, Loss: 0.0028924793004989624
Epoch 100, Loss: 0.0011298231547698379
Epoch 150, Loss: 0.9461766481399536
Epoch 200, Loss: 0.0007979070069268346
Epoch 250, Loss: 0.0008064831490628421
Epoch 300, Loss: 0.000271879427600652
Epoch 350, Loss: 0.6773098707199097
Epoch 400, Loss: 0.7474923133850098
Epoch 450, Loss: 0.0003777029050979763


In [68]:
# Prediction function
def predict_next_word(model, sentence, word2idx, idx2word):
    words = sentence.split()
    seq = [word2idx[word] for word in words if word in word2idx]
    seq = torch.tensor([seq + [0] * (max_len - len(seq))], dtype=torch.long)
    output = model(seq)
    predicted_idx = torch.argmax(output, dim=1).item()
    return idx2word[predicted_idx]

In [69]:
# Example usage
print(predict_next_word(model, "hey how", word2idx, idx2word))

you
